In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

In [0]:
base = spark.table("company_ro.gold.company_financial_summary")

window_turnover = Window.partitionBy("an").orderBy(F.col("cifra_afaceri").desc_nulls_last())
window_profit = Window.partitionBy("an").orderBy(F.col("profit_net").desc_nulls_last())
window_employees = Window.partitionBy("an").orderBy(F.col("nr_mediu_salariati").desc_nulls_last())

top_companies_by_year = (
    base
    .select(
        "an",
        "cui",
        "denumire",
        "judet",
        "localitate",
        "cod_caen_mfinante",
        "denumire_caen",
        "grupa_caen",
        "cifra_afaceri",
        "profit_net",
        "pierdere_neta",
        "nr_mediu_salariati",
        "datorii",
        "capitaluri_proprii"
    )
    .withColumn("rank_by_turnover", F.row_number().over(window_turnover))
    .withColumn("rank_by_profit", F.row_number().over(window_profit))
    .withColumn("rank_by_employees", F.row_number().over(window_employees))
    .filter(
        (F.col("rank_by_turnover") <= 100) |
        (F.col("rank_by_profit") <= 100) |
        (F.col("rank_by_employees") <= 100)
    )
)

display(top_companies_by_year.limit(100))

In [0]:
(
    top_companies_by_year
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.gold.top_companies_by_year")
)

In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows,
  MIN(rank_by_turnover) AS best_turnover_rank,
  MAX(rank_by_turnover) AS worst_turnover_rank
FROM company_ro.gold.top_companies_by_year
GROUP BY an
ORDER BY an
"""))